# RBY1 Real Robot Inference with LAP (Right Arm Only)

GR00T 실로봇 inference notebook 구조를 기반으로, **LAP 체크포인트 + WebSocket 정책 서버**를 사용하도록 구현한 버전입니다.

## 핵심 차이점 (vs GR00T)

| 항목 | GR00T | LAP |
|------|-------|-----|
| Policy | ZMQ 서버 (`Gr00tPolicy.get_action()`) | WebSocket 서버 (`WebsocketClientPolicy.infer()`) |
| Action 차원 | 16D (bimanual) | **8D** (right arm only: `right_arm[7] + right_gripper[1]`) |
| Action 형식 | dict of arrays (부위별 분리) | `actions` array `(horizon, 8)` — **이미 absolute joint target** |
| Left Arm | 모델이 직접 제어 | **현재 위치 유지** (hold) |
| 이미지 | 3캠 (head + left_wrist + right_wrist) | **2캠** (head + right_wrist) |
| State 입력 | 16D | 16D (서버 pipeline이 내부적으로 8D로 slice) |
| Normalization | Processor 내부 | 서버 pipeline (Normalize → Unnormalize) |

## 서버 실행 방법 (별도 터미널)

```bash
cd /lustre/meat124/lap
uv run scripts/serve_policy.py \
  --policy.config lap_rby1 \
  --policy.dir checkpoints/lap_rby1/<exp_name>/<step>
```

## 실행 순서
1. 설정/연결 정보 로드 + LAP Policy 서버 연결
2. 초기 자세(initial pose) 이동 — 실로봇 (safe path + head)
3. Gripper 초기화 (homing + 열기)
4. RealSense 카메라 + standalone observation 구성
5. Dry-run inference (로봇 명령 없이 출력만 확인)
6. 실로봇 Episode Loop (inference → right arm action chunk 전송)
7. 시각화 + 정리

## 주의
- LAP 정책 서버를 먼저 실행한 뒤 진행하세요.
- 실로봇 실행 전 주변 안전을 반드시 확보하세요.

## Step 1 — 라이브러리 Import 및 기본 설정

필요한 라이브러리를 불러오고, 로봇 IP, 정책 서버 주소, 카메라 시리얼, 제어 파라미터 등 전역 설정값을 정의합니다.

> **항상 가장 먼저 실행해야 하는 셀입니다.**

In [ ]:
from __future__ import annotations

import gc
import logging
import time

import cv2
import numpy as np
import pyrealsense2 as rs
import rby1_sdk as rby

logging.basicConfig(level=logging.INFO, force=True)

# ---------- User Config ----------
ROBOT_IP = "localhost:50051"
SIM_IP = "localhost:50052"

# LAP policy server (WebSocket)
POLICY_SERVER_HOST = "0.0.0.0"
POLICY_SERVER_PORT = 8000

PROMPT = "pick up the cup and place it on the plate"

# RealSense serials (head + right wrist only for LAP)
CAM_HEAD_SERIAL = "838212070714"
CAM_RIGHT_SERIAL = "838212074317"

# LAP model parameters (overridden by server metadata if available)
ACTION_DIM = 8        # right_arm[7] + right_gripper[1]
ACTION_HORIZON = 16

# Test mode — if True, simulator/single-chunk cells run
TEST_MODE = False

# Runtime/Control
EPISODE_LENGTH = 400
EXECUTE_CHUNK_SIZE = None  # None이면 predict chunk 전체 실행

ARM_COMMAND_PRIORITY = 10
ARM_MINIMUM_TIME = 10.0
USE_REMOTE_GRIPPER = True

# Initial pose (safe path + head init)
ENABLE_INITIAL_POSE = True
SAFE_INIT_PATH = True
INIT_COMMAND_PRIORITY = 10
INIT_DT = 0.05
INIT_HOLD_TIME = 0.2
ENABLE_HEAD_INIT = True
HEAD_INIT_DEG = np.array([0.0, 40.0], dtype=np.float64)
HEAD_INIT = np.deg2rad(HEAD_INIT_DEG)

# rby1 joint layout (24-DOF + grippers)
TORSO_SLICE = slice(2, 8)    # 6 joints
RIGHT_SLICE = slice(8, 15)   # 7 joints
LEFT_SLICE = slice(15, 22)   # 7 joints

print("Config loaded")
print(f"ROBOT_IP={ROBOT_IP}  SIM_IP={SIM_IP}")
print(f"POLICY_SERVER={POLICY_SERVER_HOST}:{POLICY_SERVER_PORT}")
print(f"CAMERAS head={CAM_HEAD_SERIAL} right_wrist={CAM_RIGHT_SERIAL}")
print(f"ACTION_DIM={ACTION_DIM}  ACTION_HORIZON={ACTION_HORIZON}")
print(f"PROMPT={PROMPT}")

## Step 1-2 — LAP Policy 서버 연결 + Helper 함수 정의

`openpi_client.WebsocketClientPolicy`로 LAP 정책 서버에 연결하고, observation 변환 / action 변환 helper 함수를 정의합니다.

### Action 형식 (LAP rby1)
- 서버 출력: `actions` array, shape `(action_horizon, 8)` — **이미 absolute joint target (rad)**
- Index 0-6: `right_arm` 7개 관절 (absolute position)
- Index 7: `right_gripper` (normalized 0-1)
- Left arm: 모델이 제어하지 않음 → 현재 위치 유지

In [ ]:
# ---------- Step 1-2: LAP Policy 서버 연결 ----------
from openpi_client import websocket_client_policy as _ws_client

policy = _ws_client.WebsocketClientPolicy(
    host=POLICY_SERVER_HOST,
    port=POLICY_SERVER_PORT,
)

server_meta = policy.get_server_metadata()
ACTION_DIM = int(server_meta.get("action_dim", ACTION_DIM))
ACTION_HORIZON = int(server_meta.get("action_horizon", ACTION_HORIZON))

print("LAP policy server connected.")
print(f"Server metadata: {server_meta}")
print(f"ACTION_DIM={ACTION_DIM}  ACTION_HORIZON={ACTION_HORIZON}")


# ---------- Helper: (C,H,W) uint8 -> (H,W,C) uint8 ----------
def _chw_to_hwc_uint8(image_chw: np.ndarray) -> np.ndarray:
    if image_chw.ndim == 3 and image_chw.shape[0] in (1, 3):
        image_hwc = np.transpose(image_chw, (1, 2, 0))
    else:
        image_hwc = image_chw
    if image_hwc.dtype != np.uint8:
        image_hwc = image_hwc.astype(np.uint8)
    return image_hwc


# ---------- Helper: build LAP observation dict ----------
# LAP의 RepackTransform은 dot-separated 키를 읽어서 내부 키로 변환합니다.
# 서버 pipeline이 _SelectRightArmDims로 state 16D → 8D 자동 slice 합니다.
def build_lap_input(
    state16: np.ndarray,
    head_image_hwc: np.ndarray,
    right_wrist_image_hwc: np.ndarray,
    prompt: str,
) -> dict:
    """Build observation dict for LAP policy server.

    Args:
        state16: (16,) float32 — [right_arm(7), left_arm(7), right_gripper(1), left_gripper(1)]
        head_image_hwc: (H, W, 3) uint8 — head/ego camera
        right_wrist_image_hwc: (H, W, 3) uint8 — right wrist camera
        prompt: task instruction string
    """
    return {
        "observation.images.ego_view": head_image_hwc,
        "observation.images.right_wrist": right_wrist_image_hwc,
        "observation.state": np.asarray(state16, dtype=np.float32),
        "prompt": prompt,
    }


# ---------- Helper: extract action from LAP server response ----------
def parse_lap_action(result: dict) -> np.ndarray:
    """Parse LAP server response to action array.

    Returns:
        (action_horizon, action_dim) float32 array — absolute joint targets.
        Columns: [right_arm(7), right_gripper(1)]
    """
    actions = np.asarray(result["actions"], dtype=np.float32)
    if actions.ndim == 1:
        actions = actions.reshape(1, -1)
    return actions


print("Helper functions defined.")
print("  build_lap_input()  : state + images + prompt -> LAP server input")
print("  parse_lap_action() : server response -> (T, 8) absolute target array")

## Step 2 — 실로봇 초기 자세 이동

로봇 팔을 안전한 waypoint 경로를 거쳐 데이터 수집 시작 자세로 이동시킵니다.

- **elbow 굽힘 여부**에 따라 짧은 경로(midpoint2 → initial) 또는 전체 safe path(midpoint1 → midpoint2 → initial)를 선택합니다.
- Head 초기화가 활성화되면 지정 각도로 이동합니다.
- Tool flange 12V를 공급하여 그리퍼 전원을 켭니다.

In [ ]:
# ---------- Step 2: initial pose 이동 (safe path + head) ----------
if ENABLE_INITIAL_POSE:
    TORSO_INIT_DEG = np.array([0.0, 20.0, -40.0, 35.0, 0.0, 0.0], dtype=np.float64)
    INIT_RIGHT_DEG = np.array([-24.0, -60.0, 10.0, -120.0, -60.0, 85.0, 0.0], dtype=np.float64)
    INIT_LEFT_DEG = np.array([-24.0, 60.0, -10.0, -120.0, 60.0, 85.0, 0.0], dtype=np.float64)
    TORSO_INIT = np.deg2rad(TORSO_INIT_DEG)
    INIT_RIGHT = np.deg2rad(INIT_RIGHT_DEG)
    INIT_LEFT = np.deg2rad(INIT_LEFT_DEG)

    MID1_RIGHT_DEG = np.array([70.0, -30.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)
    MID1_LEFT_DEG = np.array([70.0, 30.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)
    MID2_RIGHT_DEG = np.array([0.0, -15.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)
    MID2_LEFT_DEG = np.array([0.0, 15.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)
    MID1_RIGHT = np.deg2rad(MID1_RIGHT_DEG)
    MID1_LEFT = np.deg2rad(MID1_LEFT_DEG)
    MID2_RIGHT = np.deg2rad(MID2_RIGHT_DEG)
    MID2_LEFT = np.deg2rad(MID2_LEFT_DEG)

    ELBOW_INIT_THRESHOLD_DEG = 90.0

    def _build_body_head_cmd(torso_q, right_q, left_q, head_q, dt):
        body_builder = rby.BodyComponentBasedCommandBuilder()
        if torso_q is not None:
            body_builder = body_builder.set_torso_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(INIT_HOLD_TIME))
                .set_position(np.asarray(torso_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
        body_builder = (
            body_builder
            .set_right_arm_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(INIT_HOLD_TIME))
                .set_position(np.asarray(right_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
            .set_left_arm_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(INIT_HOLD_TIME))
                .set_position(np.asarray(left_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
        )
        cbc = rby.ComponentBasedCommandBuilder().set_body_command(body_builder)
        if head_q is not None:
            cbc = cbc.set_head_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(INIT_HOLD_TIME))
                .set_position(np.asarray(head_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
        return rby.RobotCommandBuilder().set_command(cbc)

    def _move_waypoint_stream(robot, stream, name, torso_target, right_target, left_target, head_target, duration):
        q0 = np.asarray(robot.get_state().position, dtype=np.float64)
        start_torso = q0[TORSO_SLICE].copy()
        start_right = q0[RIGHT_SLICE].copy()
        start_left = q0[LEFT_SLICE].copy()
        goal_torso = torso_target if torso_target is not None else start_torso
        steps = max(2, int(np.ceil(duration / INIT_DT)))
        torso_path = np.linspace(start_torso, goal_torso, num=steps)
        right_path = np.linspace(start_right, right_target, num=steps)
        left_path = np.linspace(start_left, left_target, num=steps)
        for i in range(steps):
            cmd = _build_body_head_cmd(torso_path[i], right_path[i], left_path[i], head_target, dt=INIT_DT)
            try:
                stream.send_command(cmd)
            except RuntimeError as exc:
                if "expired" in str(exc).lower():
                    robot.wait_for_control_ready(1000)
                    stream = robot.create_command_stream(priority=INIT_COMMAND_PRIORITY)
                    stream.send_command(cmd)
                else:
                    raise
            time.sleep(INIT_DT)
        q_now = np.asarray(robot.get_state().position, dtype=np.float64)
        print(f"[init-pose] reached {name} | torso_now[1]={q_now[3]:+.4f}, right_now[0]={q_now[8]:+.4f}, left_now[0]={q_now[15]:+.4f}")
        return stream

    robot_init = rby.create_robot(ROBOT_IP, "a") if hasattr(rby, "create_robot") else rby.create_robot_a(ROBOT_IP)
    robot_init.connect()
    assert robot_init.is_connected(), f"Failed to connect robot at {ROBOT_IP}"

    robot_init.power_on(".*")
    robot_init.servo_on(".*")
    robot_init.enable_control_manager()

    try:
        robot_init.cancel_control()
    except Exception:
        pass
    robot_init.wait_for_control_ready(1000)

    q0 = np.asarray(robot_init.get_state().position, dtype=np.float64)
    print("[init-pose] current torso:", np.round(q0[TORSO_SLICE], 3))
    print("[init-pose] current right:", np.round(q0[RIGHT_SLICE], 3))
    print("[init-pose] current left :", np.round(q0[LEFT_SLICE], 3))
    if ENABLE_HEAD_INIT:
        print("[init-pose] target head(deg):", np.round(HEAD_INIT_DEG, 2))

    right_elbow_now = float(q0[RIGHT_SLICE][3])
    left_elbow_now = float(q0[LEFT_SLICE][3])
    elbow_bent = (
        abs(right_elbow_now) > np.deg2rad(ELBOW_INIT_THRESHOLD_DEG)
        or abs(left_elbow_now) > np.deg2rad(ELBOW_INIT_THRESHOLD_DEG)
    )
    print(
        f"[init-pose] elbow check | right={np.degrees(right_elbow_now):+.1f}, "
        f"left={np.degrees(left_elbow_now):+.1f}  "
        f"-> {'near-INITIAL (bent)' if elbow_bent else 'near-ZERO (straight)'}"
    )

    if SAFE_INIT_PATH:
        if elbow_bent:
            waypoints = [
                ("midpoint2", None,       MID2_RIGHT, MID2_LEFT, 5.0),
                ("initial",   TORSO_INIT, INIT_RIGHT, INIT_LEFT, 3.0),
            ]
            print("[init-pose] near-initial detected: midpoint2 -> initial")
        else:
            waypoints = [
                ("midpoint1", None,       MID1_RIGHT, MID1_LEFT, 5.0),
                ("midpoint2", None,       MID2_RIGHT, MID2_LEFT, 5.0),
                ("initial",   TORSO_INIT, INIT_RIGHT, INIT_LEFT, 3.0),
            ]
            print("[init-pose] near-zero detected: midpoint1 -> midpoint2 -> initial")
    else:
        waypoints = [("initial", TORSO_INIT, INIT_RIGHT, INIT_LEFT, 4.0)]
        print("[init-pose] SAFE_INIT_PATH disabled: direct -> initial")

    head_target = HEAD_INIT if ENABLE_HEAD_INIT else None
    stream = robot_init.create_command_stream(priority=INIT_COMMAND_PRIORITY)
    for name, torso_target, right_target, left_target, duration in waypoints:
        print(f"[init-pose] move -> {name} (t={duration:.1f}s)")
        stream = _move_waypoint_stream(
            robot_init, stream, name=name,
            torso_target=torso_target, right_target=right_target,
            left_target=left_target, head_target=head_target, duration=duration,
        )

    time.sleep(0.2)
    q_end = np.asarray(robot_init.get_state().position, dtype=np.float64)
    err_t = float(np.linalg.norm(q_end[TORSO_SLICE] - TORSO_INIT))
    err_r = float(np.linalg.norm(q_end[RIGHT_SLICE] - INIT_RIGHT))
    err_l = float(np.linalg.norm(q_end[LEFT_SLICE] - INIT_LEFT))
    print(f"[init-pose] done | final error norm torso={err_t:.6f}, right={err_r:.6f}, left={err_l:.6f}")

    if USE_REMOTE_GRIPPER:
        for _arm in ("right", "left"):
            ok_v = robot_init.set_tool_flange_output_voltage(_arm, 12)
            print(f"[init-pose] tool flange 12V {_arm}: {'OK' if ok_v else 'FAILED'}")
        time.sleep(0.5)

    try:
        robot_init.cancel_control()
    except Exception:
        pass
    if hasattr(robot_init, "disconnect"):
        robot_init.disconnect()

    INITIAL_POSE_DONE = True
else:
    INITIAL_POSE_DONE = False
    print("[init-pose] skipped")

## Step 2-5 — Gripper 초기화 (Homing + 열기)

RemoteGripper UDP 클라이언트를 통해 그리퍼를 초기화합니다.

1. `initialize()` — 서버 ping 확인
2. `homing()` — 범위 캘리브레이션 (min_q / max_q)
3. `start()` — 제어 루프 시작
4. 완전 열기 (normalized=1.0)

In [ ]:
# ---------- Step 2-5: Gripper 초기화 (homing + 열기) ----------
if not globals().get("INITIAL_POSE_DONE", False):
    raise RuntimeError("먼저 Step 2 (initial pose) 셀을 실행하세요.")
if not globals().get("USE_REMOTE_GRIPPER", False):
    print("[gripper-init] USE_REMOTE_GRIPPER=False -> skip")
else:
    from rby1_remote_gripper import Gripper as RemoteGripper

    if "stream" in globals() and stream is not None:
        try:
            del stream
        except Exception:
            pass
        stream = None
        print("[gripper-init] 이전 stream 객체 정리 완료")

    if "gripper_obj" in globals() and gripper_obj is not None:
        try:
            gripper_obj.stop()
            print("[gripper-init] 이전 gripper 객체 stop() 완료")
        except Exception as _e:
            print(f"[gripper-init] 이전 gripper stop 무시: {_e}")
        gripper_obj = None

    print("[gripper-init] RemoteGripper 연결 중...")
    gripper_obj = RemoteGripper()
    print(f"[gripper-init] host={gripper_obj.host}  port={gripper_obj.port}")

    if gripper_obj.host is None or gripper_obj.port is None:
        raise RuntimeError(
            "[gripper-init] gripper host/port가 설정되지 않았습니다.\n"
            "  scripts/deployment/config.yaml 또는 환경변수 확인"
        )

    # 1) ping / initialize
    print("[gripper-init] ping 확인 중...")
    ok_init = gripper_obj.initialize(verbose=True)
    print(f"[gripper-init] ping 결과: {'OK' if ok_init else 'FAILED'}")
    if not ok_init:
        raise RuntimeError("[gripper-init] 그리퍼 서버 응답 없음.")

    # 2) homing
    print("[gripper-init] homing 중... (30초 이내)")
    ok_homing = gripper_obj.homing()
    if not ok_homing:
        raise RuntimeError("[gripper-init] homing 실패")

    _min_q = getattr(gripper_obj, "min_q", None)
    _max_q = getattr(gripper_obj, "max_q", None)
    if _min_q is None or _max_q is None:
        raise RuntimeError("[gripper-init] homing 성공했지만 min_q/max_q가 없음.")
    print(f"[gripper-init] homing 완료 | min_q={_min_q}  max_q={_max_q}")

    # 3) 제어 루프 시작
    print("[gripper-init] start() 호출 중...")
    gripper_obj.start()
    print("[gripper-init] start() 완료")

    # 4) 완전 열기
    _old_timeout = getattr(gripper_obj, "timeout", None)
    _fast_timeout = 3.0
    try:
        _old_timeout_f = float(_old_timeout) if _old_timeout is not None else None
    except Exception:
        _old_timeout_f = None
    if _old_timeout_f is None or _old_timeout_f > _fast_timeout:
        gripper_obj.timeout = _fast_timeout
        try:
            gripper_obj._sock.settimeout(_fast_timeout)
        except Exception:
            pass
    print(f"[gripper-init] fast timeout 적용: {_old_timeout} -> {gripper_obj.timeout}")

    # LAP는 오른쪽 그리퍼만 사용하지만, 양쪽 모두 열어둠
    print("[gripper-init] 완전 열기 명령 전송 (normalized=1.0)...")
    gripper_obj.set_normalized_target(np.array([1.0, 1.0]), wait_for_reply=True)
    time.sleep(0.5)

    try:
        _grip_state = gripper_obj.get_state()
        print(f"[gripper-init] 현재 그리퍼 상태: {_grip_state}")
    except Exception as _se:
        print(f"[gripper-init][WARN] 상태 조회 실패: {_se}")

    GRIPPER_INIT_DONE = True
    print("[gripper-init] 완료")

## Step 3 — RealSense 카메라 초기화 + 관측(Observation) 구성

`pyrealsense2`를 이용해 Head / Right Wrist 2대의 카메라를 초기화하고, `rby1_sdk`를 통해 로봇 state를 읽어 observation 함수를 정의합니다.

> GR00T는 `RBY1Environment` 래퍼를 사용하지만, LAP은 독립 구현합니다.
> LAP은 카메라 2대 (head + right_wrist)만 사용합니다.

In [ ]:
# ---------- Step 3: 카메라 초기화 + observation 함수 ----------
import matplotlib.pyplot as plt

CAM_SERIALS = {
    "head": CAM_HEAD_SERIAL,
    "right_wrist": CAM_RIGHT_SERIAL,
}

IMG_WIDTH = 640
IMG_HEIGHT = 480
RENDER_SIZE = 224  # LAP expects 224x224

# (A) RealSense 장치 진단
_ctx = rs.context()
_available = {
    dev.get_info(rs.camera_info.serial_number): dev.get_info(rs.camera_info.name)
    for dev in _ctx.query_devices()
}
del _ctx

print("-" * 60)
print(f"  [camera-check] connected devices: {len(_available)}")
for serial, name in _available.items():
    cfg_name = next((n for n, s in CAM_SERIALS.items() if s == serial), "unknown")
    mark = "OK" if serial in CAM_SERIALS.values() else "INFO"
    print(f"  {mark:4s} {cfg_name:>12} | {name} | S/N: {serial}")

_not_found = [(n, s) for n, s in CAM_SERIALS.items() if s not in _available]
if _not_found:
    for n, s in _not_found:
        print(f"  MISSING {n}: {s}")
    raise RuntimeError("Some configured cameras are not detected. Check USB connections.")
print(f"  OK: all configured cameras detected ({len(CAM_SERIALS)})")
print("-" * 60)

# (B) 기존 파이프라인 정리
if "_rs_pipelines" in globals():
    for _p in _rs_pipelines.values():
        try:
            _p.stop()
        except Exception:
            pass
    print("[cleanup] 기존 RS 파이프라인 정리 완료")

gc.collect()
time.sleep(1.0)

# (C) RealSense 파이프라인 시작
_rs_pipelines: dict[str, rs.pipeline] = {}
_rs_configs: dict[str, rs.config] = {}

for cam_name, serial in CAM_SERIALS.items():
    pipe = rs.pipeline()
    cfg = rs.config()
    cfg.enable_device(serial)
    cfg.enable_stream(rs.stream.color, IMG_WIDTH, IMG_HEIGHT, rs.format.bgr8, 30)
    pipe.start(cfg)
    _rs_pipelines[cam_name] = pipe
    _rs_configs[cam_name] = cfg
    print(f"[camera] {cam_name} ({serial}) started: {IMG_WIDTH}x{IMG_HEIGHT}@30fps")

# (D) 로봇 연결 (state 읽기용)
robot_obs = rby.create_robot(ROBOT_IP, "a") if hasattr(rby, "create_robot") else rby.create_robot_a(ROBOT_IP)
robot_obs.connect()
assert robot_obs.is_connected(), f"Failed to connect robot at {ROBOT_IP}"
print(f"[robot] connected for observation: {ROBOT_IP}")


# ---------- observation helper ----------
def get_observation() -> dict:
    """Read cameras + robot state and return raw observation dict.

    Returns dict with keys:
        head_image:        (H, W, 3) uint8 BGR -> RGB, resized to RENDER_SIZE
        right_wrist_image: (H, W, 3) uint8 BGR -> RGB, resized to RENDER_SIZE
        state:             (16,) float32 — full joint state from rby1_sdk
    """
    images = {}
    for cam_name, pipe in _rs_pipelines.items():
        frames = pipe.wait_for_frames()
        color_frame = frames.get_color_frame()
        if not color_frame:
            raise RuntimeError(f"[obs] no color frame from {cam_name}")
        bgr = np.asanyarray(color_frame.get_data())
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        rgb_resized = cv2.resize(rgb, (RENDER_SIZE, RENDER_SIZE), interpolation=cv2.INTER_AREA)
        images[cam_name] = rgb_resized

    q = np.asarray(robot_obs.get_state().position, dtype=np.float64)
    # Build 16D state: [right_arm(7), left_arm(7), right_gripper(1), left_gripper(1)]
    right_arm = q[RIGHT_SLICE]    # 7 DOF
    left_arm = q[LEFT_SLICE]      # 7 DOF

    # gripper normalized values
    if USE_REMOTE_GRIPPER and "gripper_obj" in globals() and gripper_obj is not None:
        try:
            _gnorm = gripper_obj.get_normalized_target()
            right_grip = float(_gnorm[0])
            left_grip = float(_gnorm[1])
        except Exception:
            right_grip = 1.0
            left_grip = 1.0
    else:
        right_grip = 1.0
        left_grip = 1.0

    state16 = np.concatenate([
        right_arm,                        # [0:7]   right arm
        left_arm,                         # [7:14]  left arm
        np.array([right_grip]),           # [14]    right gripper
        np.array([left_grip]),            # [15]    left gripper
    ]).astype(np.float32)

    return {
        "head_image": images["head"],
        "right_wrist_image": images["right_wrist"],
        "state": state16,
    }


# (E) 첫 관측 테스트 + 시각화
obs = get_observation()
print(f"[obs] head_image: {obs['head_image'].shape}, right_wrist_image: {obs['right_wrist_image'].shape}")
print(f"[obs] state: {obs['state'].shape}")
print(f"[obs] state (rad): {np.round(obs['state'], 4)}")

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, (title, img) in zip(axes, [("head", obs["head_image"]), ("right_wrist", obs["right_wrist_image"])]):
    ax.imshow(img)
    ax.set_title(title, fontsize=11)
    ax.axis("off")
fig.suptitle("Initial Observation (LAP)", fontsize=13)
plt.tight_layout()
plt.show()

ENV_SETUP_DONE = True
print("[setup] 완료: 카메라 + 로봇 state 관측 확인")

## Step 4 — Dry-run Inference (로봇 명령 없음)

LAP inference를 1회 수행하고 출력만 확인합니다. 로봇에는 아무 명령도 보내지 않습니다.

**출력값 해석:**
- `parse_lap_action()` 반환값은 **absolute joint target (rad)** — shape `(action_horizon, 8)`
- 현재 state와 가깝게 나오면 정상 (delta가 작을수록 현재 위치 근처)
- Index 0-6: right arm joints, Index 7: right gripper (normalized 0-1)

In [ ]:
# ---------- Step 4: Dry-run inference ----------
if not globals().get("ENV_SETUP_DONE", False):
    raise RuntimeError("먼저 Step 3 셀을 실행하세요.")

obs_dry = get_observation()
lap_input = build_lap_input(
    state16=obs_dry["state"],
    head_image_hwc=obs_dry["head_image"],
    right_wrist_image_hwc=obs_dry["right_wrist_image"],
    prompt=PROMPT,
)

result_dry = policy.infer(lap_input)
actions_dry = parse_lap_action(result_dry)  # (T, 8)

current_state = obs_dry["state"]
print(f"[dry-run] action chunk shape: {actions_dry.shape}")
print(f"[dry-run] current state (rad): {np.round(current_state, 4)}")
print()
print(f"[dry-run] absolute right_arm[0] : {np.round(actions_dry[0, :7], 4)}")
print(f"[dry-run] absolute right_arm[-1]: {np.round(actions_dry[-1, :7], 4)}")
print(f"[dry-run] gripper[0]={actions_dry[0, 7]:.4f}, gripper[-1]={actions_dry[-1, 7]:.4f}")
print()

# 현재 state와의 차이 (de-facto relative delta) 확인
current_right_arm = current_state[:7]
delta_arm = actions_dry[:, :7] - current_right_arm[None, :]
print("[dry-run] implied delta from current right_arm state:")
for t in range(actions_dry.shape[0]):
    d_norm = float(np.linalg.norm(delta_arm[t]))
    if t == 0 or t == actions_dry.shape[0] - 1 or (t + 1) % 4 == 0:
        print(f"  step {t:2d}: delta_norm={d_norm:.4f} rad | gripper={actions_dry[t, 7]:.3f}")

---
## Step 5 — 실로봇 Inference Episode 루프 실행

LAP inference → action chunk 실행을 반복하며 전체 에피소드를 수행합니다.

- **`EPISODE_LENGTH`**: 총 실행 스텝 수
- **`EXECUTE_CHUNK_SIZE`**: inference 1회당 실제 전송 스텝 수 (None = 전체 chunk)
- **`REPLAY_DT`**: 각 step 전송 간격(초)

### LAP action 형식
- `parse_lap_action()` → shape `(action_horizon, 8)` — **absolute joint target (rad)**
- Index 0-6: right arm → `set_right_arm_command()`
- Index 7: right gripper (normalized 0-1)
- **Left arm은 모델 출력 없음** → 현재 위치 유지 (`q[LEFT_SLICE]`)

In [ ]:
# ---------- Step 5: LAP policy action replay — episode loop ----------
if not globals().get("ENV_SETUP_DONE", False):
    raise RuntimeError("먼저 Step 3 셀을 실행하세요.")
if not globals().get("INITIAL_POSE_DONE", False):
    raise RuntimeError("먼저 Step 2 셀(초기 자세 이동)을 실행하세요.")

# --------------------------------------------------
# 재생 파라미터
# --------------------------------------------------
REPLAY_DT        = 0.1
REPLAY_PRIORITY  = ARM_COMMAND_PRIORITY
REPLAY_HOLD_TIME = 0.2

# --------------------------------------------------
# 로봇 연결
# --------------------------------------------------
robot = rby.create_robot(ROBOT_IP, "a") if hasattr(rby, "create_robot") else rby.create_robot_a(ROBOT_IP)
robot.connect()
assert robot.is_connected(), f"Failed to connect robot at {ROBOT_IP}"

robot.power_on(".*")
robot.servo_on(".*")
robot.reset_fault_control_manager()
if not robot.enable_control_manager():
    robot.disconnect()
    raise RuntimeError("[real-replay] Failed to enable control manager")

q_init     = np.asarray(robot.get_state().position, dtype=np.float64)
ep_start_r = q_init[RIGHT_SLICE].copy()
ep_start_l = q_init[LEFT_SLICE].copy()
torso_hold = q_init[TORSO_SLICE].copy()
print(f"[real-replay] EPISODE_LENGTH={EPISODE_LENGTH}, EXECUTE_CHUNK_SIZE={EXECUTE_CHUNK_SIZE}")
print(f"[real-replay] episode start | right={np.round(ep_start_r, 4)}")

stream      = robot.create_command_stream(priority=REPLAY_PRIORITY)
total_steps = 0
chunk_idx   = 0
send_errors = 0

# --------------------------------------------------
# 로깅 버퍼
# --------------------------------------------------
_log_cmd_right    = []
_log_cmd_grip     = []
_log_actual_right = []
_log_actual_step  = []
_log_chunk_start  = []
_IMG_KEYS = ["head_image", "right_wrist_image"]
_log_obs_images   = []

# --------------------------------------------------
# Episode loop: observe -> infer -> send right arm (left arm hold)
# --------------------------------------------------
while total_steps < EPISODE_LENGTH:
    remaining = EPISODE_LENGTH - total_steps

    # 1) observation + inference
    obs_ep = get_observation()

    _img_snap = {"step": total_steps}
    for _k in _IMG_KEYS:
        if _k in obs_ep:
            _img_snap[_k] = obs_ep[_k].copy()
    _log_obs_images.append(_img_snap)

    lap_input_ep = build_lap_input(
        state16=obs_ep["state"],
        head_image_hwc=obs_ep["head_image"],
        right_wrist_image_hwc=obs_ep["right_wrist_image"],
        prompt=PROMPT,
    )
    result_ep = policy.infer(lap_input_ep)
    actions_ep = parse_lap_action(result_ep)  # (T, 8) absolute

    right_targets = actions_ep[:, :7]     # right arm absolute targets
    right_gripper_out = actions_ep[:, 7]  # right gripper normalized

    # 2) left arm hold position (모델이 left arm을 출력하지 않음)
    q_now = np.asarray(robot.get_state().position, dtype=np.float64)
    left_hold = q_now[LEFT_SLICE].copy()
    torso_hold = q_now[TORSO_SLICE].copy()

    chunk_size    = actions_ep.shape[0]
    execute_size  = EXECUTE_CHUNK_SIZE if EXECUTE_CHUNK_SIZE is not None else chunk_size
    steps_to_send = min(execute_size, remaining)
    total_cmd_norm = float(np.linalg.norm(right_targets[min(steps_to_send-1, chunk_size-1)] - right_targets[0]))

    _log_chunk_start.append(total_steps)
    print(
        f"[real-replay] chunk {chunk_idx+1} | "
        f"steps {total_steps+1}~{total_steps+steps_to_send}/{EPISODE_LENGTH} "
        f"(execute {steps_to_send}/{chunk_size}) | cmd_norm={total_cmd_norm:.4f}"
    )

    # 3) chunk 전송 — right arm from model, left arm hold current
    for i in range(steps_to_send):
        idx = min(i, chunk_size - 1)  # clamp if execute > chunk

        cmd = rby.RobotCommandBuilder().set_command(
            rby.ComponentBasedCommandBuilder().set_body_command(
                rby.BodyComponentBasedCommandBuilder()
                .set_torso_command(
                    rby.JointPositionCommandBuilder()
                    .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(REPLAY_HOLD_TIME))
                    .set_position(torso_hold.astype(np.float64))
                    .set_minimum_time(REPLAY_DT)
                )
                .set_right_arm_command(
                    rby.JointPositionCommandBuilder()
                    .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(REPLAY_HOLD_TIME))
                    .set_position(right_targets[idx].astype(np.float64))
                    .set_minimum_time(REPLAY_DT)
                )
                .set_left_arm_command(
                    rby.JointPositionCommandBuilder()
                    .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(REPLAY_HOLD_TIME))
                    .set_position(left_hold.astype(np.float64))
                    .set_minimum_time(REPLAY_DT)
                )
            )
        )

        try:
            stream.send_command(cmd)
        except RuntimeError as exc:
            send_errors += 1
            if "expired" in str(exc).lower():
                print(f"[real-replay][WARN] stream expired at step {total_steps}")
                robot.wait_for_control_ready(1000)
                stream = robot.create_command_stream(priority=REPLAY_PRIORITY)
                stream.send_command(cmd)
            else:
                raise

        # gripper command (right only, left stays open)
        if USE_REMOTE_GRIPPER and globals().get("gripper_obj") is not None:
            try:
                gripper_obj.set_normalized_target(
                    np.array([float(right_gripper_out[idx]), 1.0]),  # left=1.0 (open)
                    wait_for_reply=False,
                )
            except Exception as _ge:
                print(f"[real-replay][WARN] gripper send failed: {_ge}")

        _log_cmd_right.append(right_targets[idx].copy())
        _log_cmd_grip.append(float(right_gripper_out[idx]))

        if i == 0 or (i + 1) % 10 == 0 or (i + 1) == steps_to_send:
            q_diag  = np.asarray(robot.get_state().position, dtype=np.float64)
            err_r   = float(np.linalg.norm(right_targets[idx] - q_diag[RIGHT_SLICE]))
            moved_r = float(np.linalg.norm(q_diag[RIGHT_SLICE] - ep_start_r))
            _log_actual_right.append(q_diag[RIGHT_SLICE].copy())
            _log_actual_step.append(total_steps + i)
            print(
                f"  step {total_steps+i+1:4d}/{EPISODE_LENGTH} "
                f"| tracking_err(r={err_r:.4f}) "
                f"| moved(r={moved_r:.4f}) "
                f"| gripper={right_gripper_out[idx]:+.3f}"
            )

        time.sleep(REPLAY_DT)

    total_steps += steps_to_send
    chunk_idx   += 1

# --------------------------------------------------
# 최종 결과
# --------------------------------------------------
time.sleep(0.5)
q_after    = np.asarray(robot.get_state().position, dtype=np.float64)
obs_norm_r = float(np.linalg.norm(q_after[RIGHT_SLICE] - ep_start_r))

print(f"\n[real-replay] === episode done ===")
print(f"[real-replay] total_steps={total_steps}, chunks={chunk_idx}, send_errors={send_errors}")
print(f"[real-replay] ep_start right : {np.round(ep_start_r, 4)}")
print(f"[real-replay] q_after  right : {np.round(q_after[RIGHT_SLICE], 4)}")
print(f"[real-replay] total movement norm | right={obs_norm_r:.4f}")
print(f"[real-replay] log: {len(_log_cmd_right)} steps, {len(_log_chunk_start)} chunks")

try:
    robot.cancel_control()
except Exception:
    pass
if hasattr(robot, "disconnect"):
    robot.disconnect()

---
## Action 로그 플롯 — 관절 궤적 및 Chunk 경계 분석

Episode loop 실행 후 `_log_*` 버퍼를 기반으로 Right Arm 7 관절 + Gripper 궤적을 시각화합니다.

| 그래프 | 내용 |
|--------|------|
| 전체 궤적 | 관절별 commanded vs actual (deg) |
| Step-to-step delta | chunk 경계 spike 확인 |

In [ ]:
# ---------- Action chunk 로그 시각화 (Right Arm Only) ----------
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

if "_log_cmd_right" not in globals() or len(_log_cmd_right) == 0:
    print("[plot] 로그 데이터 없음 — episode loop를 먼저 실행하세요.")
else:
    cmd_r    = np.array(_log_cmd_right)
    grip_r   = np.array(_log_cmd_grip)
    act_r    = np.array(_log_actual_right) if _log_actual_right else np.empty((0, 7))
    act_x    = np.array(_log_actual_step,  dtype=int)
    c_starts = np.array(_log_chunk_start,  dtype=int)

    N = cmd_r.shape[0]
    steps_x = np.arange(N)

    print(f"[plot] steps: {N}  |  chunks: {len(c_starts)}  |  actual samples: {len(act_x)}")

    # (A) Right Arm 전체 궤적 (7 joints + gripper)
    fig, axes = plt.subplots(2, 4, figsize=(20, 8), sharex=True)
    fig.suptitle("Right Arm Trajectory (deg) — LAP", fontsize=13, fontweight="bold")
    axes_flat = axes.flatten()
    for j in range(7):
        ax = axes_flat[j]
        ax.plot(steps_x, np.degrees(cmd_r[:, j]), color="steelblue", lw=1.2, label="commanded")
        if len(act_x) > 0:
            ax.scatter(act_x, np.degrees(act_r[:, j]), color="tomato", s=18, zorder=5, label="actual")
        for cs in c_starts:
            ax.axvline(x=cs, color="dimgray", linestyle="--", lw=0.8, alpha=0.7)
        ax.set_title(f"R_J{j}", fontsize=10)
        ax.set_ylabel("deg", fontsize=8)
        ax.tick_params(labelsize=7)
        ax.grid(True, alpha=0.3)
    ax_g = axes_flat[7]
    ax_g.plot(steps_x, grip_r, color="darkorange", lw=1.2)
    for cs in c_starts:
        ax_g.axvline(x=cs, color="dimgray", linestyle="--", lw=0.8, alpha=0.7)
    ax_g.set_title("R_Gripper (0=close, 1=open)", fontsize=10)
    ax_g.set_ylabel("value", fontsize=8)
    ax_g.tick_params(labelsize=7)
    ax_g.grid(True, alpha=0.3)
    for ax in axes[1]:
        ax.set_xlabel("step", fontsize=8)
    handles = [
        mpatches.Patch(color="steelblue", label="commanded"),
        mpatches.Patch(color="tomato",    label="actual (sampled)"),
        plt.Line2D([0], [0], color="dimgray", linestyle="--", label="chunk start"),
    ]
    fig.legend(handles=handles, loc="lower center", ncol=3, fontsize=9, bbox_to_anchor=(0.5, -0.01))
    plt.tight_layout(rect=[0, 0.04, 1, 1])
    plt.show()

    # (B) Step-to-step delta norm
    delta_r = np.linalg.norm(np.diff(cmd_r, axis=0), axis=1)

    fig2, ax2 = plt.subplots(figsize=(16, 4))
    fig2.suptitle("Step-to-step Command Delta (L2 norm, rad) — Right Arm", fontsize=11, fontweight="bold")
    ax2.plot(np.arange(N - 1), delta_r, color="steelblue", lw=0.9, label="right arm delta")
    for k, cs in enumerate(c_starts[1:]):
        ax2.axvline(x=cs - 1, color="red", linestyle="--", lw=1.5, alpha=0.85,
                    label="chunk boundary" if k == 0 else None)
    ax2.set_ylabel("delta norm (rad)", fontsize=9)
    ax2.set_xlabel("step", fontsize=9)
    ax2.grid(True, alpha=0.3)
    ax2.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    print("\n[plot] === Step delta stats (chunk boundary vs internal) ===")
    in_mask = np.ones(len(delta_r), dtype=bool)
    for cs in c_starts[1:]:
        if 0 < cs - 1 < len(in_mask):
            in_mask[cs - 1] = False
    b_delta = delta_r[~in_mask]
    i_delta = delta_r[in_mask]
    print(
        f"  Right: in-chunk avg={i_delta.mean():.4f} rad | "
        f"boundary avg={b_delta.mean() if len(b_delta) else 0:.4f} rad | "
        f"boundary max={b_delta.max()  if len(b_delta) else 0:.4f} rad"
    )

## Policy 입력 RGB 이미지 프레임 확인

Episode loop 실행 중 매 inference 시점에 캡처된 카메라 프레임을 시각화합니다.

- **행**: 카메라 종류 (Head / Right Wrist)
- **열**: inference 시점 (chunk 인덱스)

In [ ]:
# ---------- Policy 입력 RGB 이미지 프레임 시각화 ----------
import matplotlib.pyplot as plt

if "_log_obs_images" not in globals() or len(_log_obs_images) == 0:
    print("[img-plot] 로그 데이터 없음 — episode loop를 먼저 실행하세요.")
else:
    cam_keys = [k for k in _IMG_KEYS if k in _log_obs_images[0]]
    n_snaps  = len(_log_obs_images)
    n_cams   = len(cam_keys)

    MAX_COLS = 12
    step_indices = list(range(0, n_snaps, max(1, n_snaps // MAX_COLS)))[:MAX_COLS]

    fig, axes = plt.subplots(n_cams, len(step_indices),
                             figsize=(3 * len(step_indices), 3 * n_cams))
    if n_cams == 1:
        axes = axes[np.newaxis, :]
    if len(step_indices) == 1:
        axes = axes[:, np.newaxis]

    for col, si in enumerate(step_indices):
        snap = _log_obs_images[si]
        for row, ck in enumerate(cam_keys):
            ax = axes[row, col]
            if ck in snap:
                ax.imshow(snap[ck])
            ax.axis("off")
            if col == 0:
                ax.set_ylabel(ck, fontsize=9, rotation=0, labelpad=80, va="center")
            if row == 0:
                ax.set_title(f"chunk {si}\nstep={snap.get('step', '?')}", fontsize=8)

    fig.suptitle("Policy Input RGB Frames (per inference) — LAP", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()
    print(f"[img-plot] {n_snaps} inference snapshots, showing {len(step_indices)}")

# [Initial Pose] (episode 후 복귀용)
위의 Step 2 코드와 동일합니다. Episode 완료 후 초기 자세로 복귀할 때 사용합니다.

In [ ]:
# ---------- [Reset] Initial Pose 복귀 ----------
# Step 2와 동일한 로직. Episode 후 다시 초기 자세로 이동.
if ENABLE_INITIAL_POSE:
    TORSO_INIT_DEG = np.array([0.0, 20.0, -40.0, 35.0, 0.0, 0.0], dtype=np.float64)
    INIT_RIGHT_DEG = np.array([-24.0, -60.0, 10.0, -120.0, -60.0, 85.0, 0.0], dtype=np.float64)
    INIT_LEFT_DEG = np.array([-24.0, 60.0, -10.0, -120.0, 60.0, 85.0, 0.0], dtype=np.float64)
    TORSO_INIT = np.deg2rad(TORSO_INIT_DEG)
    INIT_RIGHT = np.deg2rad(INIT_RIGHT_DEG)
    INIT_LEFT = np.deg2rad(INIT_LEFT_DEG)

    MID2_RIGHT = np.deg2rad(np.array([0.0, -15.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64))
    MID2_LEFT = np.deg2rad(np.array([0.0, 15.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64))

    robot_reset = rby.create_robot(ROBOT_IP, "a") if hasattr(rby, "create_robot") else rby.create_robot_a(ROBOT_IP)
    robot_reset.connect()
    assert robot_reset.is_connected()
    robot_reset.power_on(".*")
    robot_reset.servo_on(".*")
    robot_reset.enable_control_manager()
    try:
        robot_reset.cancel_control()
    except Exception:
        pass
    robot_reset.wait_for_control_ready(1000)

    head_target = HEAD_INIT if ENABLE_HEAD_INIT else None
    stream_reset = robot_reset.create_command_stream(priority=INIT_COMMAND_PRIORITY)

    waypoints = [
        ("midpoint2", None,       MID2_RIGHT, MID2_LEFT, 5.0),
        ("initial",   TORSO_INIT, INIT_RIGHT, INIT_LEFT, 3.0),
    ]
    for name, torso_target, right_target, left_target, duration in waypoints:
        print(f"[reset] move -> {name} (t={duration:.1f}s)")
        stream_reset = _move_waypoint_stream(
            robot_reset, stream_reset, name=name,
            torso_target=torso_target, right_target=right_target,
            left_target=left_target, head_target=head_target, duration=duration,
        )

    time.sleep(0.2)
    q_end = np.asarray(robot_reset.get_state().position, dtype=np.float64)
    print(f"[reset] done | right err={np.linalg.norm(q_end[RIGHT_SLICE]-INIT_RIGHT):.6f}")

    if USE_REMOTE_GRIPPER:
        for _arm in ("right", "left"):
            robot_reset.set_tool_flange_output_voltage(_arm, 12)
        time.sleep(0.5)

    try:
        robot_reset.cancel_control()
    except Exception:
        pass
    if hasattr(robot_reset, "disconnect"):
        robot_reset.disconnect()
    INITIAL_POSE_DONE = True
else:
    print("[reset] ENABLE_INITIAL_POSE=False -> skip")

# [Gripper Re-init] (episode 후 재초기화용)
위의 Step 2-5 코드와 동일합니다. Episode 완료 후 그리퍼를 다시 homing/열기할 때 사용합니다.

In [ ]:
# ---------- [Reset] Gripper 재초기화 ----------
if not globals().get("USE_REMOTE_GRIPPER", False):
    print("[gripper-reinit] USE_REMOTE_GRIPPER=False -> skip")
else:
    from rby1_remote_gripper import Gripper as RemoteGripper

    if "gripper_obj" in globals() and gripper_obj is not None:
        try:
            gripper_obj.stop()
        except Exception:
            pass
        gripper_obj = None

    gripper_obj = RemoteGripper()
    ok_init = gripper_obj.initialize(verbose=True)
    if not ok_init:
        raise RuntimeError("[gripper-reinit] 서버 응답 없음")

    ok_homing = gripper_obj.homing()
    if not ok_homing:
        raise RuntimeError("[gripper-reinit] homing 실패")
    print(f"[gripper-reinit] homing 완료 | min_q={gripper_obj.min_q} max_q={gripper_obj.max_q}")

    gripper_obj.start()
    gripper_obj.timeout = 3.0
    try:
        gripper_obj._sock.settimeout(3.0)
    except Exception:
        pass

    gripper_obj.set_normalized_target(np.array([1.0, 1.0]), wait_for_reply=True)
    time.sleep(0.5)
    GRIPPER_INIT_DONE = True
    print("[gripper-reinit] 완료 (양쪽 열림)")

---
## Step 6 — 환경 및 정리

에피소드 종료 후 카메라, 로봇, 그리퍼 리소스를 안전하게 정리합니다.

In [ ]:
# ---------- Step 6: 정리 ----------
# 카메라 파이프라인 정리
if "_rs_pipelines" in globals():
    for _name, _p in _rs_pipelines.items():
        try:
            _p.stop()
            print(f"[cleanup] camera {_name} stopped")
        except Exception as _e:
            print(f"[cleanup] camera {_name} stop 실패: {_e}")
    _rs_pipelines = {}

# 로봇 연결 정리
if "robot_obs" in globals() and robot_obs is not None:
    try:
        if hasattr(robot_obs, "disconnect"):
            robot_obs.disconnect()
        print("[cleanup] robot_obs disconnected")
    except Exception as _e:
        print(f"[cleanup] robot_obs disconnect 실패: {_e}")
    robot_obs = None

# 그리퍼 정리
if "gripper_obj" in globals() and gripper_obj is not None:
    try:
        gripper_obj.stop()
        print("[cleanup] gripper stopped")
    except Exception as _e:
        print(f"[cleanup] gripper stop 실패: {_e}")
    gripper_obj = None

ENV_SETUP_DONE = False
INITIAL_POSE_DONE = False
GRIPPER_INIT_DONE = False
print("[cleanup] done")